LOADING DATA AND SELECTING TARGET AND FEATURES

In [1]:

# load and prepare dataset

import pandas as pd
import os 


In [2]:
file_path = "../../data/processed/housing_cleaned.csv"

In [3]:
#reading csv file


df= pd.read_csv(file_path)

In [4]:
print("Number of rows and columns:",df.shape)

Number of rows and columns: (13766, 30)


In [5]:
df.head

<bound method NDFrame.head of          Price  Bedrooms  Bathrooms  Garage  Furnished  Amenity_Dstv  \
0      30940.0         4          4       0          1             1   
1      30940.0         2          2       0          1             1   
2      40221.0         3          3       2          0             0   
3      58785.0         4          4       2          0             1   
4      57238.0         3          2       2          0             0   
...        ...       ...        ...     ...        ...           ...   
13761  41768.0         4          5       7          1             1   
13762  23205.0         4          5       5          0             1   
13763  38674.0         3          4       3          0             0   
13764  26299.0         5          6       6          0             1   
13765  23205.0         4          5       6          0             0   

       Amenity_Internet  Amenity_Pets allowed  Amenity_Refrigerator  \
0                     1           

In [ ]:
# feature selection
# target column is price and predictors are the selected features 

selected_features=['Bedrooms','Bathrooms','luxuryFeatures','comfortFeatures','utilityFeatures','connectivityFeatures','exSpace' ]

target= 'LogPrice'

In [7]:
x=df[selected_features]
y=df[target]


In [8]:
df[selected_features].head()

,Bedrooms,Bathrooms,luxuryFeatures,comfortFeatures,utilityFeatures,connectivityFeatures,exSpace
0,4,4,1,6,2,2,0
1,2,2,2,5,4,2,0
2,3,3,2,5,3,0,0
3,4,4,2,5,4,2,1
4,3,2,2,5,3,0,0


TRAIN-TEST SPLIT

In [ ]:
# the goal is to see how the model will perform on new, unseen data
# training set teaches the model
# testing set evaluates how well the model generalizes

from sklearn.model_selection import train_test_split
import numpy as np

x_train, x_test, y_train, y_test = train_test_split(x , y , test_size=0.2, random_state= 42)

print(f"Training data shape for features (x_train):{x_train.shape}")
print(f"Testing data shape for features (x_test):{x_test.shape}")
print(f"Training data shape for target (y_train): {y_train.shape}")
print(f"Testing data shape for target (y_test): {y_test.shape}")


Training data shape for features (x_train):(11012, 7)
Testing data shape for features (x_test):(2754, 7)
Training data shape for target (y_train): (11012,)
Testing data shape for target (y_test): (2754,)


FEATURE SCALING(FEATURE ENGINEERING)

In [ ]:
# feature scaling(standardization) is a type of preprocessing(part of feature engineering)
# makes the model more stable and improves accuracy

# ml models like linear regression assume that all features are on a similar scale 
# if one feature has a value of 5000 and another has a value of 20, the model will give importance to the larger scale feature
#  prevent the imbalance, use StandardScaler, which transforms each feature

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

#  fit the scaler(transformers) on training data and transform both test and training data

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)




In [ ]:
#imputation means handling missing values in your dataset.  use median because it is robust against outliers

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')

#fit on training data only 
x_train_imputed = imputer.fit_transform(x_train_scaled)

#transform test data
x_test_imputed = imputer.transform(x_test_scaled)

x_train_scaled = x_train_imputed

x_test_scaled = x_test_imputed





In [ ]:
# initially, the model only explained 18.3% of the variation in housing prices, i have to improve the model by adding polynomial features

from sklearn.preprocessing import PolynomialFeatures

# initialize PolynomialFeatures

poly = PolynomialFeatures(degree =2 , include_bias = False)

#fit and transform training data

x_train_poly = poly.fit_transform(x_train_scaled)

#transform test data using the fitted polynomial features
x_test_poly = poly.transform(x_test_scaled)

x_train_scaled = x_train_poly
x_test_scaled = x_test_poly


MODEL TRAINING

In [13]:
#import model
from sklearn.linear_model import LinearRegression

#initialize model
linear_model = LinearRegression()
linear_model.fit(x_train_scaled, y_train)



,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [14]:
# predictions on the test set

y_pred_log = linear_model.predict(x_test_scaled)

# inverse transform predictions for metrics
y_pred = np.expm1(y_pred_log)



In [ ]:
#inverse transform y_test to compare with y_pred 
y_test_original = np.expm1(y_test)

In [16]:
#evaluating models performance using regression metrics
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error


In [17]:
#r2 tells how much of the variation in price is explained by the model( close to 1 is better)
r2 = r2_score(y_test_original,y_pred)


In [18]:
#RMSE gives an idea of error magnitude in the same unit as price
rmse = np.sqrt(mean_squared_error(y_test_original,y_pred))


In [19]:
#MAE is the average absolute error in predictions
mae = mean_absolute_error(y_test_original,y_pred)


model performance

In [20]:
print("R² score is", r2)
print("RMSE is", rmse)
print("MAE is", mae)

R² score is 0.24546710648344128
RMSE is 23548.743714401542
MAE is 14401.71528043317


In [ ]:

#import pandas as pd


#results = pd.DataFrame({
    #'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
    #'R²': [r2, rf_r2, xgb_r2],
    #'RMSE': [rmse, rf_rmse, xgb_rmse],
    #'MAE': [mae, rf_mae, xgb_mae]
#})

#print("\nModel Comparison:")
#print(results)


Model Comparison:
               Model        R²          RMSE           MAE
0  Linear Regression  0.245467  23548.743714  14401.715280
1      Random Forest  0.367189      0.828848      0.628576
2            XGBoost  0.411620      0.799221      0.605820
